# Phase 3 — Programmatic DAG Generation
**Author:** Leo Yu  
**Project:** Foundations of Data Science — Final Project  
**Topic:** Does preventative physiotherapy reduce overuse injuries in athletes?

---

## Purpose
This notebook generates the **Directed Acyclic Graph (DAG)** for our causal model.  
The DAG is the visual statement of what we believe causes what, and it justifies the structure of the statistical model used in Phases 1 and 2.

Running this notebook produces `plot_dag.png`, which is embedded in the master submission notebook (Section 2).

## The Causal Model — In Plain English

We are studying three variables:

| Symbol | Variable | Role |
|--------|----------|------|
| **X** | Training Load (Intensity × Duration) | **Confounder** |
| **T** | Recovery Score (proxy for preventative physiotherapy) | **Treatment** |
| **Y** | Injury Status (1 = injured, 0 = not injured) | **Outcome** |

Three causal arrows exist:

1. **X → T** — Athletes who train harder seek more recovery (training load drives the treatment).
2. **X → Y** — Higher training load directly raises injury risk (the confounder's direct effect on the outcome).
3. **T → Y** — Recovery affects injury risk (this is the causal effect we want to estimate).

Because X has a path into both T and Y, ignoring X would bias our estimate of T → Y. The statistical model in Phase 1 controls for X by including it as a regressor.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────────────
# We use networkx to define the graph structure and matplotlib to render it.
# These are both standard libraries that come with the conda-env-fnds-py environment.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

print(f"✅ networkx version : {nx.__version__}")
print(f"✅ matplotlib loaded")

In [ ]:
# ── Define the DAG ──────────────────────────────────────────────────────────────────────
# Build a directed graph with three nodes (X, T, Y) and the three causal arrows.
# Using DiGraph (not Graph) is what makes this *directed* — the arrows have direction.

G = nx.DiGraph()

# Add the three nodes with descriptive labels stored as node attributes
G.add_node("X", role="Confounder", full_name="Training Load")
G.add_node("T", role="Treatment",  full_name="Recovery Score")
G.add_node("Y", role="Outcome",    full_name="Injury Status")

# Add the three causal edges. Each tuple (A, B) means an arrow A → B.
G.add_edge("X", "T")  # training load drives recovery behavior
G.add_edge("X", "Y")  # training load directly affects injury risk (confounding path)
G.add_edge("T", "Y")  # recovery affects injury risk — this is the causal effect we want

# Verify the graph is acyclic (no loops) — this is required for a valid DAG
assert nx.is_directed_acyclic_graph(G), "Graph contains a cycle — not a valid DAG!"

print("Graph nodes:", list(G.nodes(data=True)))
print("Graph edges:", list(G.edges()))
print("Is DAG?    :", nx.is_directed_acyclic_graph(G))

In [ ]:
# ── Draw the DAG ────────────────────────────────────────────────────────────────────────
# We pick fixed node positions so the diagram looks the same every time.
# X is placed at the top-center (the confounder "sits above" both other variables).
# T is bottom-left (treatment), Y is bottom-right (outcome).
# This layout makes the confounding path X→T and X→Y visually obvious.

positions = {
    "X": (0.5, 1.0),   # top-center
    "T": (0.0, 0.0),   # bottom-left
    "Y": (1.0, 0.0),   # bottom-right
}

# Color-code the nodes by role for clarity
node_colors = {
    "X": "#4C8DBF",  # blue   = confounder
    "T": "#5DAE6F",  # green  = treatment
    "Y": "#E07A5F",  # red    = outcome
}

fig, ax = plt.subplots(figsize=(9, 6))

# Draw nodes one at a time so each can have its own color
for node, (x, y) in positions.items():
    nx.draw_networkx_nodes(
        G, positions,
        nodelist=[node],
        node_color=node_colors[node],
        node_size=4500,
        edgecolors="black",
        linewidths=2,
        ax=ax
    )

# Draw the arrows. arrowstyle='-|>' gives a clean filled arrowhead.
nx.draw_networkx_edges(
    G, positions,
    arrowstyle="-|>",
    arrowsize=28,
    edge_color="black",
    width=2.0,
    node_size=4500,        # tells the renderer to start arrows at the node boundary
    connectionstyle="arc3,rad=0.0",
    ax=ax
)

# Draw the symbol labels (X, T, Y) inside each node
nx.draw_networkx_labels(
    G, positions,
    labels={n: n for n in G.nodes()},
    font_size=22,
    font_weight="bold",
    font_color="white",
    ax=ax
)

# Add full-name annotations next to each node so readers don't have to remember symbols
annotations = {
    "X": (0.5, 1.18, "Training Load (Confounder)"),
    "T": (0.0, -0.18, "Recovery Score (Treatment)"),
    "Y": (1.0, -0.18, "Injury Status (Outcome)"),
}
for _, (x, y, text) in annotations.items():
    ax.text(x, y, text, ha="center", va="center", fontsize=11, fontweight="bold")

# Label the causal-effect-of-interest arrow (T → Y)
ax.text(0.5, -0.08, r"causal effect of interest: $\beta_T$",
        ha="center", va="center", fontsize=10, style="italic", color="#444")

# Legend
legend_handles = [
    mpatches.Patch(color="#4C8DBF", label="Confounder (X)"),
    mpatches.Patch(color="#5DAE6F", label="Treatment (T)"),
    mpatches.Patch(color="#E07A5F", label="Outcome (Y)"),
]
ax.legend(handles=legend_handles, loc="upper right", frameon=True, fontsize=10)

ax.set_title("Causal DAG — Sports Injury Project",
             fontsize=14, fontweight="bold", pad=20)
ax.set_xlim(-0.35, 1.35)
ax.set_ylim(-0.35, 1.35)
ax.axis("off")

plt.tight_layout()
plt.savefig("./plot_dag.png", dpi=150, bbox_inches="tight")
plt.show()

print("✅ DAG saved to plot_dag.png")

## How to Read This DAG

- **The arrow we want to estimate:** `T → Y`. Its strength is captured by the parameter $\beta_T$ in the statistical model.
- **The reason we cannot just regress Y on T alone:** the path `T ← X → Y` is a *backdoor path*. It creates a non-causal correlation between T and Y because both share the common cause X.
- **How the statistical model handles this:** by including X as a regressor in the logistic equation
$$\text{logit}(p_i) = \alpha + \beta_T \cdot T_i + \beta_X \cdot X_i$$
we close the backdoor and $\beta_T$ becomes an unbiased estimate of the causal effect of recovery on injury risk.

This DAG matches the model that Tilak validated in Phase 1 and that Advait will fit on real data in Phase 2.